# Deep Learning Architectures

## Loading dataset

- MNIST dataset --> 70,000 samples of handwritten digits in grayscale image [28 X 28 pixels].
- x_train --> images
- y_train --> digit class (0-9)

In [1]:
from tensorflow.keras import datasets

(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
# Normalize pixel values (0–255 → 0–1)
x_train = x_train[..., None] / 255.0
x_test = x_test[..., None] / 255.0

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape) 

2025-10-18 16:19:42.921440: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-18 16:20:01.739553: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-18 16:20:13.245506: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


(60000, 28, 28, 1) (60000,) (10000, 28, 28, 1) (10000,)


## CNN

**Key Notes**:
- Input shape --> should match with the shape of the input sample
- There are lot of hyperparameters involved in **Conv2D** layer,
  - filters --> these can be thought of as number of output channels (or feature maps).
  - kernel_size --> larger kernels can capture broader patterns, while smaller capture local patterns more.
  - strides --> step size when scanning the kernel over the input. Higher stride leads to smaller output feature map.
  - padding --> 'same' keeps output size same as input, 'valid' reduces it.
  - activation --> 
  - Each filter (of size 3) has --> 3x3*1(num of channel) + 1 bias = 10 parameters. For 32 filters, there will be 320 parameters.
- In **MaxPooling()**
  - pool_size --> how much down sampling is needed
  - strides
  - padding --> whether to pad the borders or not
- In **Dense()** layer
  - units
  - activation
  - kernel_initializer --> weight initialization. This affects convergence.
  - kernel_regularizer
- Following are **training hyperparameters**,
  - optimizer, learning rate, loss function, batch size, epochs, validation split

In [ ]:
from tensorflow.keras import layers, models, Input

inputs = Input(shape=(28, 28, 1))             # each input sample is 28x28 pixel, and gray scale (i.e., 1 channel)
x = layers.Conv2D(32, 3, activation='relu')(inputs)  # 32 different, 3x3 filters
x = layers.MaxPooling2D(pool_size=(2,2), strides=None, padding='valid')(x)
x = layers.Conv2D(64, 3, activation='relu')(x)    # 64 filters of size 3x3
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

2025-10-18 16:20:59.967856: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 7744)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       495,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 515,146 (1.97 MB)

 Trainable params: 515,146 (1.97 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [4]:
# train the model
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/5


2025-10-18 16:21:12.054827: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 169344000 exceeds 10% of free system memory.


844/844 ━━━━━━━━━━━━━━━━━━━━ 32s 36ms/step - accuracy: 0.9571 - loss: 0.1424 - val_accuracy: 0.9852 - val_loss: 0.0529
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 29s 34ms/step - accuracy: 0.9868 - loss: 0.0422 - val_accuracy: 0.9890 - val_loss: 0.0404
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 25s 29ms/step - accuracy: 0.9908 - loss: 0.0285 - val_accuracy: 0.9890 - val_loss: 0.0409
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 24s 28ms/step - accuracy: 0.9939 - loss: 0.0187 - val_accuracy: 0.9903 - val_loss: 0.0379
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 41s 29ms/step - accuracy: 0.9953 - loss: 0.0146 - val_accuracy: 0.9910 - val_loss: 0.0307


In [5]:
# evaluate on test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

 30/313 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9927 - loss: 0.0200

2025-10-18 16:26:17.243286: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 31360000 exceeds 10% of free system memory.


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9911 - loss: 0.0269
Test accuracy: 0.9911
